In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import read_csv_from_blob
from src.dataset import assemble_dataset
from src.train import train_per_horizon_models, predict_next_day

STORAGE_ACCOUNT = "sagreekdamdevweu"
GATE_CLOSURE_HOUR = 12
TARGET_DAY_ATHENS = pd.Timestamp("2026-05-16", tz="Europe/Athens")

print(f"Target prediction day (Athens): {TARGET_DAY_ATHENS.date()}")

Target prediction day (Athens): 2026-05-16


In [2]:
print("Loading processed/ data from blob...")

dam = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "dam_prices/full.csv")
load = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "load_forecast/full.csv")
renewable = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "renewable_forecast/full.csv")

print(f"DAM:        {len(dam)} rows, {dam.index.min()} → {dam.index.max()}")
print(f"Load:       {len(load)} rows, {load.index.min()} → {load.index.max()}")
print(f"Renewable:  {len(renewable)} rows, {renewable.index.min()} → {renewable.index.max()}")

Loading processed/ data from blob...
DAM:        29520 rows, 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00
Load:       29567 rows, 2022-12-31 22:00:00+00:00 → 2026-05-16 20:00:00+00:00
Renewable:  29567 rows, 2022-12-31 22:00:00+00:00 → 2026-05-16 20:00:00+00:00


In [3]:
renewable

,solar,wind_onshore
2022-12-31 22:00:00+00:00,0.0,495.0
2022-12-31 23:00:00+00:00,0.0,518.0
2023-01-01 00:00:00+00:00,0.0,560.0
2023-01-01 01:00:00+00:00,0.0,603.0
2023-01-01 02:00:00+00:00,0.0,649.0
...,...,...
2026-05-16 16:00:00+00:00,485.0,2745.0
2026-05-16 17:00:00+00:00,47.5,2707.5
2026-05-16 18:00:00+00:00,0.0,2612.5
2026-05-16 19:00:00+00:00,0.0,2505.0


In [4]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

Training data (inner join): 29520 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00


In [7]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

lgbm_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 20,
    "random_state": 42,
    "verbose": -1,
}

result = train_per_horizon_models(
    prices_train,
    exog=exog_train,
    gate_closure_hour=GATE_CLOSURE_HOUR,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=1,
    test_days=30,  # ← FULL DATASET
    lgbm_params=lgbm_params,
)

print("\nTraining complete!")
print(f"Models trained: {len(result.models)}")
print(result.summary())


Training data (inner join): 29520 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00

Training complete!
Models trained: 24
TrainResult Summary
  Models trained:    24 per-horizon LightGBM models
  Features:          38
  Gate closure:      12:00
  Overall test MAE:  17.35 EUR/MWh
  Overall test RMSE: 23.23 EUR/MWh
  Best horizon:      h= 8 (MAE 11.14)
  Worst horizon:     h=18 (MAE 27.36)


In [8]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

lgbm_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 20,
    "random_state": 42,
    "verbose": -1,
}

result = train_per_horizon_models(
    prices_train,
    exog=exog_train,
    gate_closure_hour=GATE_CLOSURE_HOUR,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=1,
    test_days=0,  # ← FULL DATASET
    lgbm_params=lgbm_params,
)

print("\nTraining complete!")
print(f"Models trained: {len(result.models)}")
print(f"(Metrics are NaN since no test set)")


Training data (inner join): 29520 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00

Training complete!
Models trained: 24
(Metrics are NaN since no test set)


In [9]:
################################
# Forecast for May 16
################################

prices, exog = assemble_dataset(dam, load, renewable, join="outer")

# Forecast time = noon Athens on May 13 (today)
forecast_time_athens = TARGET_DAY_ATHENS - pd.Timedelta(days=1) + pd.Timedelta(hours=GATE_CLOSURE_HOUR)
forecast_time_utc = forecast_time_athens.tz_convert("UTC")
print(f"Forecast time (Athens): {forecast_time_athens}")
print(f"Forecast time (UTC):    {forecast_time_utc}")

if forecast_time_utc not in prices.index:
    print(f"WARNING: forecast_time not in prices index!")
    closest = prices.index[prices.index <= forecast_time_utc][-1]
    print(f"Closest available: {closest}")
else:
    print(f"forecast_time present in index ✓")

forecast_may16 = predict_next_day(
    result, prices, exog=exog, forecast_time=forecast_time_utc
)

print(f"\nForecast for May 16:")
print(forecast_may16.to_frame(name="predicted_eur_mwh"))

Forecast time (Athens): 2026-05-15 12:00:00+03:00
Forecast time (UTC):    2026-05-15 09:00:00+00:00
forecast_time present in index ✓

Forecast for May 16:
                           predicted_eur_mwh
2026-05-15 21:00:00+00:00         102.147717
2026-05-15 22:00:00+00:00          95.541209
2026-05-15 23:00:00+00:00          91.379362
2026-05-16 00:00:00+00:00          94.388123
2026-05-16 01:00:00+00:00          83.160885
2026-05-16 02:00:00+00:00          86.661147
2026-05-16 03:00:00+00:00         118.758559
2026-05-16 04:00:00+00:00         116.250643
2026-05-16 05:00:00+00:00          83.835628
2026-05-16 06:00:00+00:00          48.333708
2026-05-16 07:00:00+00:00          15.162685
2026-05-16 08:00:00+00:00           5.275025
2026-05-16 09:00:00+00:00           1.397759
2026-05-16 10:00:00+00:00          -7.796441
2026-05-16 11:00:00+00:00           1.809565
2026-05-16 12:00:00+00:00          47.973995
2026-05-16 13:00:00+00:00          91.497030
2026-05-16 14:00:00+00:00         1

In [26]:
# Fetch published DAM for May 16 (Athens)##################
from src.data_loader import fetch_dam_prices
from src.processing import ensure_hourly

print("Fetching published DAM for May 16 from ENTSO-E...")

# Fetch a window covering May 14 (Athens local)
start = pd.Timestamp("2026-05-16", tz="Europe/Athens")
end = pd.Timestamp("2026-05-18", tz="Europe/Athens")

actual_may16 = fetch_dam_prices(start=start, end=end, country_code="GR")
actual_may16 = ensure_hourly(actual_may16)

# Convert index to UTC so it matches our forecast format
actual_may16.index = actual_may16.index.tz_convert("UTC")

print(f"Fetched {len(actual_may16)} rows")
print(f"Range: {actual_may16.index.min()} → {actual_may16.index.max()}")
print()
print(actual_may16.to_string())

Fetching published DAM for May 16 from ENTSO-E...
Fetched 25 rows
Range: 2026-05-15 21:00:00+00:00 → 2026-05-16 21:00:00+00:00

2026-05-15 21:00:00+00:00    139.5400
2026-05-15 22:00:00+00:00    127.3375
2026-05-15 23:00:00+00:00    125.9925
2026-05-16 00:00:00+00:00    125.7875
2026-05-16 01:00:00+00:00    121.5750
2026-05-16 02:00:00+00:00    126.0000
2026-05-16 03:00:00+00:00    113.7200
2026-05-16 04:00:00+00:00    109.3900
2026-05-16 05:00:00+00:00    104.0750
2026-05-16 06:00:00+00:00     72.1600
2026-05-16 07:00:00+00:00      7.4700
2026-05-16 08:00:00+00:00      7.0750
2026-05-16 09:00:00+00:00     11.1250
2026-05-16 10:00:00+00:00      7.0150
2026-05-16 11:00:00+00:00      7.3900
2026-05-16 12:00:00+00:00      6.1525
2026-05-16 13:00:00+00:00      6.9275
2026-05-16 14:00:00+00:00     20.9975
2026-05-16 15:00:00+00:00     65.3425
2026-05-16 16:00:00+00:00    109.4750
2026-05-16 17:00:00+00:00    108.6425
2026-05-16 18:00:00+00:00    122.1875
2026-05-16 19:00:00+00:00    121.060

In [27]:
# Fetch published DAM for May 15 (Athens)
from src.data_loader import fetch_dam_prices
from src.processing import ensure_hourly

print("Fetching published DAM for May 15 from ENTSO-E...")

# Fetch a window covering May 14 (Athens local)
start = pd.Timestamp("2026-05-15", tz="Europe/Athens")
end = pd.Timestamp("2026-05-16", tz="Europe/Athens")

actual_may15 = fetch_dam_prices(start=start, end=end, country_code="GR")
actual_may15 = ensure_hourly(actual_may15)

# Convert index to UTC so it matches our forecast format
actual_may15.index = actual_may15.index.tz_convert("UTC")

print(f"Fetched {len(actual_may15)} rows")
print(f"Range: {actual_may15.index.min()} → {actual_may15.index.max()}")
print()
print(actual_may15.to_string())

Fetching published DAM for May 15 from ENTSO-E...
Fetched 25 rows
Range: 2026-05-14 21:00:00+00:00 → 2026-05-15 21:00:00+00:00

2026-05-14 21:00:00+00:00    135.9900
2026-05-14 22:00:00+00:00    133.6000
2026-05-14 23:00:00+00:00    126.2275
2026-05-15 00:00:00+00:00    123.4375
2026-05-15 01:00:00+00:00    124.4975
2026-05-15 02:00:00+00:00    131.9225
2026-05-15 03:00:00+00:00    134.7600
2026-05-15 04:00:00+00:00    142.1900
2026-05-15 05:00:00+00:00    127.6950
2026-05-15 06:00:00+00:00    106.9700
2026-05-15 07:00:00+00:00      0.0075
2026-05-15 08:00:00+00:00      0.0000
2026-05-15 09:00:00+00:00      0.0000
2026-05-15 10:00:00+00:00      0.0000
2026-05-15 11:00:00+00:00      0.0000
2026-05-15 12:00:00+00:00      0.0000
2026-05-15 13:00:00+00:00      0.0000
2026-05-15 14:00:00+00:00     26.0125
2026-05-15 15:00:00+00:00    127.5225
2026-05-15 16:00:00+00:00    141.2375
2026-05-15 17:00:00+00:00    157.3425
2026-05-15 18:00:00+00:00    165.6000
2026-05-15 19:00:00+00:00    147.080

In [28]:
naive = actual_may15
naive.index = naive.index + pd.Timedelta(days=1)
naive

2026-05-15 21:00:00+00:00    135.9900
2026-05-15 22:00:00+00:00    133.6000
2026-05-15 23:00:00+00:00    126.2275
2026-05-16 00:00:00+00:00    123.4375
2026-05-16 01:00:00+00:00    124.4975
2026-05-16 02:00:00+00:00    131.9225
2026-05-16 03:00:00+00:00    134.7600
2026-05-16 04:00:00+00:00    142.1900
2026-05-16 05:00:00+00:00    127.6950
2026-05-16 06:00:00+00:00    106.9700
2026-05-16 07:00:00+00:00      0.0075
2026-05-16 08:00:00+00:00      0.0000
2026-05-16 09:00:00+00:00      0.0000
2026-05-16 10:00:00+00:00      0.0000
2026-05-16 11:00:00+00:00      0.0000
2026-05-16 12:00:00+00:00      0.0000
2026-05-16 13:00:00+00:00      0.0000
2026-05-16 14:00:00+00:00     26.0125
2026-05-16 15:00:00+00:00    127.5225
2026-05-16 16:00:00+00:00    141.2375
2026-05-16 17:00:00+00:00    157.3425
2026-05-16 18:00:00+00:00    165.6000
2026-05-16 19:00:00+00:00    147.0800
2026-05-16 20:00:00+00:00    144.0000
2026-05-16 21:00:00+00:00    148.7200
Freq: h, Name: price_eur_mwh, dtype: float64

In [34]:
import plotly.graph_objects as go

# df has a DatetimeIndex and columns: forecast_may14, actual_may14

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=forecast_may16.index,
    y=forecast_may16,
    mode='lines+markers',
    name='forecast_may16',
    line=dict(dash='dash')
))

fig.add_trace(go.Scatter(
    x=actual_may16.index,
    y=actual_may16,
    mode='lines+markers',
    name='actual_may16'
))

fig.add_trace(go.Scatter(
    x=naive.index,
    y=naive,
    mode='lines+markers',
    name='naive'
))

fig.update_layout(
    title='Forecast vs Actual — May 16',
    xaxis_title='Time',
    yaxis_title='Value',
    hovermode='x unified',
    xaxis=dict(tickformat='%H:%M')  # or '%Y-%m-%d %H:%M' for full datetime
)

fig.show()

In [31]:
df = pd.concat([forecast_may16, actual_may16], axis=1)
df = pd.concat([df, naive.rename("naive")], axis=1)
df['error'] = np.abs(df['prediction_eur_mwh'] - df['price_eur_mwh'])
df['naive_error'] = np.abs(df['naive'] - df['price_eur_mwh'])
df

,prediction_eur_mwh,price_eur_mwh,naive,error,naive_error
2026-05-15 21:00:00+00:00,102.147717,139.5400,135.9900,37.392283,3.5500
2026-05-15 22:00:00+00:00,95.541209,127.3375,133.6000,31.796291,6.2625
2026-05-15 23:00:00+00:00,91.379362,125.9925,126.2275,34.613138,0.2350
2026-05-16 00:00:00+00:00,94.388123,125.7875,123.4375,31.399377,2.3500
2026-05-16 01:00:00+00:00,83.160885,121.5750,124.4975,38.414115,2.9225
2026-05-16 02:00:00+00:00,86.661147,126.0000,131.9225,39.338853,5.9225
2026-05-16 03:00:00+00:00,118.758559,113.7200,134.7600,5.038559,21.0400
2026-05-16 04:00:00+00:00,116.250643,109.3900,142.1900,6.860643,32.8000
2026-05-16 05:00:00+00:00,83.835628,104.0750,127.6950,20.239372,23.6200
2026-05-16 06:00:00+00:00,48.333708,72.1600,106.9700,23.826292,34.8100


In [32]:
df['error'].mean()

np.float64(26.648021540259624)

In [33]:
df['naive_error'].mean()

np.float64(17.7047)

In [15]:
forecast_df = forecast_may14.to_frame(name="predicted_price_eur_mwh")
forecast_df.index = forecast_df.index.tz_convert("UTC")
forecast_df.to_csv("local_may14_forecast.csv")
print("Saved local prediction to local_may14_forecast.csv")

Saved local prediction to local_may14_forecast.csv
